# InfoQuant-02 — Trois copies de Werner : la frontière de distillabilité (C#)

[Version Python](InfoQuant-02-Trois-Copies-Werner-Distillabilite-Python.ipynb) · [Information quantique](README.md)

Quand la transposée partielle d'un état de Werner devient négative, peut-on déjà extraire de l'intrication à partir de trois copies ? On compare le seuil NPT à un témoin explicite de rang deux, puis on montre ce qu'un échantillonnage permet — et ne permet pas — de conclure. Le carnet Python donne la provenance du résultat et le détail de sa preuve. Ici, le calcul exact emploie seulement la bibliothèque standard .NET.

## 1. Calculer sans prendre de racine

Pour un opérateur $C$ non nul, évaluer la forme sur $C/\|C\|_F$ revient à diviser par $\|C\|_F^2$. Cette homogénéité permet de garder des matrices à coefficients entiers et des résultats rationnels, même lorsque le vecteur normalisé contient des racines carrées.

In [1]:
using System;
using System.Globalization;
using System.Linq;
using System.Numerics;

(BigInteger N, BigInteger D) Reduire(BigInteger n, BigInteger d)
{
    if (d <= 0) throw new ArgumentOutOfRangeException(nameof(d));
    var g = BigInteger.GreatestCommonDivisor(n, d);
    return (n/g, d/g);
}
void Verifier(bool condition, string message)
{
    if (!condition) throw new InvalidOperationException(message);
}
string Fraction((BigInteger N, BigInteger D) x) => x.D == 1 ? x.N.ToString() : $"{x.N}/{x.D}";
string Decimal((BigInteger N, BigInteger D) x) => ((double)x.N/(double)x.D).ToString("0.000000", CultureInfo.InvariantCulture);
Verifier(Reduire(-12, 18) == (-2, 3), "Réduction de fraction");
Console.WriteLine($"-12/18 = {Fraction(Reduire(-12, 18))}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

-12/18 = -2/3


## 2. État de Werner et seuil NPT

Le retournement $F$ a les valeurs propres $+1$ et $-1$, de multiplicités respectives $d(d+1)/2$ et $d(d-1)/2$. Sa transposée partielle vaut $d$ fois un projecteur de rang un. Ces deux décompositions donnent exactement le poids antisymétrique, la trace et la plus petite valeur propre après transposition, sans solveur spectral.

In [2]:
(BigInteger N, BigInteger D) Lambda(int d, BigInteger p, BigInteger q) => Reduire((q+p)*(d-1), 2*(d*q-p));
(BigInteger N, BigInteger D) Alpha(int d, BigInteger n, BigInteger m) => Reduire(2*n*d-(d-1)*m, 2*n+(d-1)*m);
foreach (var (p,q) in new (long,long)[]{(1,10),(1,5),(1,3),(2,5),(1,2),(999999997,3000000000),(1000000003,3000000000)})
{
    const int d = 3;
    BigInteger z = d*(d*(BigInteger)q-p);
    BigInteger plus = d*(d+1)/2, moins = d*(d-1)/2;
    Verifier(plus*(q-p)+moins*(q+p) == z, "Trace de Werner");
    var poids = Lambda(d,p,q);
    Verifier(Reduire(moins*(q+p),z) == poids, "Poids par multiplicité");
    Verifier(Alpha(d,poids.N,poids.D) == Reduire(p,q), "Conversion inverse");
    var gamma = Reduire(q-d*p,z);
    Console.WriteLine($"alpha={Fraction(Reduire(p,q)),-23} trace=1  lambda={Fraction(poids),-22} NPT={gamma.N < 0}");
}
Verifier(Lambda(3,1,2) == Reduire(3*(3-1),4*3-2), "Seuil de Dür pour d=3");

alpha=1/10                    trace=1  lambda=11/29                  NPT=False


alpha=1/5                     trace=1  lambda=3/7                    NPT=False


alpha=1/3                     trace=1  lambda=1/2                    NPT=False


alpha=2/5                     trace=1  lambda=7/13                   NPT=True


alpha=1/2                     trace=1  lambda=3/5                    NPT=True


alpha=999999997/3000000000    trace=1  lambda=3999999997/8000000003  NPT=False


alpha=1000000003/3000000000   trace=1  lambda=4000000003/7999999997  NPT=True


## 3. Contracter les indices sur les trois sites

Pour chaque sous-ensemble $S$, on somme les entrées dont les indices ligne et colonne coïncident sur les sites tracés, puis on calcule $\|\operatorname{Tr}_S C\|_F^2$. Le bit de poids fort désigne le premier site. Le masque vide donne le dénominateur de normalisation. Un témoin diagonal contrôle les huit contractions, notamment la trace totale.

In [3]:
BigInteger[] TraceNormes((int Re, int Im)[,] c, int d)
{
    int n = d*d*d;
    Verifier(c.GetLength(0) == n && c.GetLength(1) == n, "Dimension du tenseur");
    var norms = new BigInteger[8];
    for (int mask = 0; mask < 8; mask++)
    {
        int side = 1;
        for (int bit = 0; bit < 3; bit++)
            if ((mask & (1 << bit)) == 0) side *= d;
        var reduced = new int[n];
        var traced = new int[n];
        for (int i = 0; i < n; i++)
        {
            int index = i;
            for (int bit = 0; bit < 3; bit++, index /= d)
                if ((mask & (1 << bit)) == 0) reduced[i] = reduced[i]*d + index%d;
                else traced[i] = traced[i]*d + index%d;
        }
        var re = new long[side,side];
        var im = new long[side,side];
        for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
        {
            if (traced[i] != traced[j]) continue;
            re[reduced[i],reduced[j]] += c[i,j].Re;
            im[reduced[i],reduced[j]] += c[i,j].Im;
        }
        for (int i = 0; i < side; i++)
        for (int j = 0; j < side; j++)
            norms[mask] += (BigInteger)re[i,j]*re[i,j] + (BigInteger)im[i,j]*im[i,j];
    }
    return norms;
}
(BigInteger N, BigInteger D) QNormalise(BigInteger[] t, int p, int q)
{
    BigInteger n = 0;
    for (int mask = 0; mask < 8; mask++)
    {
        int k = BitOperations.PopCount((uint)mask);
        n += t[mask]*BigInteger.Pow(-p,k)*BigInteger.Pow(q,3-k);
    }
    return Reduire(n,t[0]*BigInteger.Pow(q,3));
}
(BigInteger N, BigInteger D) Courbe(int p, int q) => Reduire((BigInteger)(q-2*p)*(q-p)*(q-p),BigInteger.Pow(q,3));
var diagonal = new (int Re,int Im)[27,27];
diagonal[0,0] = diagonal[9,9] = (1,0);
var normes = TraceNormes(diagonal,3);
Verifier(normes.SequenceEqual(new BigInteger[]{2,2,2,2,4,4,4,4}), "Huit traces du témoin diagonal");
Console.WriteLine(string.Join(", ", normes));

2, 2, 2, 2, 4, 4, 4, 4


### Vérifier les contractions sur un même opérateur complexe

Le témoin diagonal ne sollicite pas les parties imaginaires. Construisons aussi $C=XY^*$ à partir de deux colonnes complexes à supports disjoints. Sur ces mêmes entrées, `tr_sites` du carnet Python donne les huit normes ci-dessous, dans l'ordre des masques de 0 à 7. Ce test compare les contractions elles-mêmes, indépendamment des générateurs aléatoires.

In [4]:
var fixture = new (int Re, int Im)[27,27];
var couples = new[] {
    (Indices: new[] {0,9}, X: new[] {(1,2),(2,-1)}, Y: new[] {(0,1),(2,0)}),
    (Indices: new[] {1,3}, X: new[] {(1,0),(0,1)}, Y: new[] {(2,0),(0,3)})
};
foreach (var (indices,x,y) in couples)
for (int i = 0; i < 2; i++)
for (int j = 0; j < 2; j++)
{
    var (a,b) = x[i];
    var (e,f) = y[j];
    fixture[indices[i],indices[j]] = (a*e+b*f,b*e-a*f);
}
var fixtureNormes = TraceNormes(fixture,3);
Verifier(fixtureNormes.SequenceEqual(new BigInteger[]{76,71,75,95,71,82,94,130}), "Contrôle croisé des huit contractions complexes");
var fixtureQ = QNormalise(fixtureNormes,3,10);
Verifier(fixtureQ == Reduire(1589,3800), "Forme normalisée du témoin complexe");
Console.WriteLine(string.Join(", ",fixtureNormes));
Console.WriteLine($"q(3/10) = {Fraction(fixtureQ)} ≈ {Decimal(fixtureQ)}");

76, 71, 75, 95, 71, 82, 94, 130


q(3/10) = 1589/3800 ≈ 0.418158


## 4. Témoin produit et courbe exacte

Le plan est engendré par les deux premiers vecteurs de base. On reprend les directions de la version Python : $(1,1,1)$ et $(1,0,-1)$, prolongées par des zéros. Les facteurs de normalisation disparaissent dans le quotient. Les deux blocs diagonaux sont identiques et chacun a rang un : le témoin a donc rang deux. Sa valeur est le produit des trois contributions $(1-2\alpha)(1-\alpha)^2$.

In [5]:
(int Re,int Im)[,] TemoinProduit(int d)
{
    var u = new int[d]; u[0] = u[1] = u[2] = 1;
    var v = new int[d]; v[0] = 1; v[2] = -1;
    int n = d*d*d;
    var c = new (int Re,int Im)[n,n];
    for (int s = 0; s < 2; s++)
    for (int i = 0; i < d*d; i++)
    for (int j = 0; j < d*d; j++)
        c[s*d*d+i,s*d*d+j] = (u[i/d]*u[j/d]*v[i%d]*v[j%d],0);
    return c;
}
var t3 = TraceNormes(TemoinProduit(3),3);
Verifier(t3[0] == 72, "Norme avant normalisation : 2*3²*2²");
Verifier(QNormalise(t3,3,10) == Reduire(49,250), "Produit des trois valeurs à alpha=0,3");
for (int p = 0; p <= 10; p++)
{
    var valeur = QNormalise(t3,p,20);
    Verifier(valeur == Courbe(p,20), "Courbe du témoin produit");
    Console.WriteLine($"alpha={Fraction(Reduire(p,20)),-5} q={Fraction(valeur),-12} ({Decimal(valeur)})");
}

alpha=0     q=1            (1.000000)


alpha=1/20  q=3249/4000    (0.812250)


alpha=1/10  q=81/125       (0.648000)


alpha=3/20  q=2023/4000    (0.505750)


alpha=1/5   q=48/125       (0.384000)


alpha=1/4   q=9/32         (0.281250)


alpha=3/10  q=49/250       (0.196000)


alpha=7/20  q=507/4000     (0.126750)


alpha=2/5   q=9/125        (0.072000)


alpha=9/20  q=121/4000     (0.030250)


alpha=1/2   q=0            (0.000000)


## 5. Échantillonner à rang deux, sans confondre recherche et preuve

Un tirage fabrique deux paires de colonnes et forme $XY^*$ : le rang est au plus deux par construction. Ici les parties réelles et imaginaires sont des entiers tirés uniformément dans $[-8,8]$, avec une graine fixe. Les mêmes budgets et valeurs de $\alpha$ sont utilisés, mais ce ne sont ni les mêmes matrices ni la même loi que les tirages gaussiens du Python. On compare les fractions par produit en croix. Dans chaque dimension, on réutilise le même lot à tous les $\alpha$ : les normes des huit traces ne dépendent pas de $\alpha$.

In [6]:
(int Re,int Im)[,] Tirage(int n, Random rng)
{
    var x = new (int Re,int Im)[n,2];
    var y = new (int Re,int Im)[n,2];
    for (int i = 0; i < n; i++)
    for (int k = 0; k < 2; k++)
    {
        x[i,k] = (rng.Next(-8,9),rng.Next(-8,9));
        y[i,k] = (rng.Next(-8,9),rng.Next(-8,9));
    }
    var c = new (int Re,int Im)[n,n];
    for (int i = 0; i < n; i++)
    for (int j = 0; j < n; j++)
    for (int k = 0; k < 2; k++)
    {
        var (a,b) = x[i,k]; var (e,f) = y[j,k];
        c[i,j].Re += a*e+b*f;
        c[i,j].Im += b*e-a*f;
    }
    return c;
}
int Comparer((BigInteger N,BigInteger D) a, (BigInteger N,BigInteger D) b) => (a.N*b.D).CompareTo(b.N*a.D);
(BigInteger N,BigInteger D)[] MinimaEchantillon(int d, int nombre, int graine, (int P,int Q)[] grille)
{
    var rng = new Random(graine);
    var minima = Enumerable.Repeat((N: new BigInteger(2),D: BigInteger.One),grille.Length).ToArray();
    for (int i = 0; i < nombre;)
    {
        var t = TraceNormes(Tirage(d*d*d,rng),d);
        if (t[0].IsZero) continue;
        i++;
        for (int j = 0; j < grille.Length; j++)
        {
            var valeur = QNormalise(t,grille[j].P,grille[j].Q);
            if (Comparer(valeur,minima[j]) < 0) minima[j] = valeur;
        }
    }
    return minima;
}
void AfficherEchantillon(int d, int n, int seed, (int P,int Q)[] grille)
{
    var minima = MinimaEchantillon(d,n,seed,grille);
    for (int j = 0; j < grille.Length; j++)
    {
        var (p,q) = grille[j];
        var refVal = Courbe(p,q);
        if (d <= 4) Verifier(Comparer(minima[j],refVal) >= 0, "Tirage sous la borne démontrée");
        Console.WriteLine($"alpha={Fraction(Reduire(p,q)),-6} tirage={Decimal(minima[j])}  témoin={Decimal(refVal)}");
    }
}
Console.WriteLine("d=3, 4000 matrices de rang au plus deux");
AfficherEchantillon(3,4000,123,new[]{(1,5),(3,10),(7,20),(2,5),(9,20),(1,2)});

d=3, 4000 matrices de rang au plus deux


alpha=1/5    tirage=0.777426  témoin=0.384000


alpha=3/10   tirage=0.677000  témoin=0.196000


alpha=7/20   tirage=0.628998  témoin=0.126750


alpha=2/5    tirage=0.582661  témoin=0.072000


alpha=9/20   tirage=0.537990  témoin=0.030250


alpha=1/2    tirage=0.494983  témoin=0.000000


## 6. Dimension quatre

Le minimum démontré est le même qu'en dimension trois. La marge entre le meilleur tirage et le témoin illustre la difficulté à approcher les directions spéciales du minimiseur. Ce n'est pas le nuage de points qui établit la borne.

In [7]:
Console.WriteLine("d=4, 2000 matrices de rang au plus deux");
var t4 = TraceNormes(TemoinProduit(4),4);
var grille4 = new[]{(1,4),(7,20),(9,20),(1,2)};
foreach (var (p,q) in grille4) Verifier(QNormalise(t4,p,q) == Courbe(p,q), "Témoin d=4");
AfficherEchantillon(4,2000,456,grille4);

d=4, 2000 matrices de rang au plus deux


alpha=1/4    tirage=0.806420  témoin=0.281250


alpha=7/20   tirage=0.733609  témoin=0.126750


alpha=9/20   tirage=0.663121  témoin=0.030250


alpha=1/2    tirage=0.628682  témoin=0.000000


## 7. Dimension cinq : témoin explicite, minimum inconnu

La borne $37/80$ assure l'indistillabilité à trois copies jusque-là. Entre $37/80$ (exclu) et $1/2$ (inclus), un tirage positif ne démontre rien sur le minimum. À droite de $1/2$, le témoin produit négatif suffit à certifier la distillabilité. Son annulation au seuil ne prouve pas qu'aucun autre opérateur ne donne une valeur négative.

In [8]:
var grille5 = new[]{(3,10),(2,5),(37,80),(12,25),(1,2),(11,20)};
var t5 = TraceNormes(TemoinProduit(5),5);
var best5 = MinimaEchantillon(5,800,789,grille5);
string Zone(int p, int q) => 80*p <= 37*q ? "borne démontrée" : 2*p <= q ? "zone ouverte" : "témoin négatif";
for (int i = 0; i < grille5.Length; i++)
{
    var (p,q) = grille5[i];
    var temoin = QNormalise(t5,p,q);
    Verifier(temoin == Courbe(p,q) && temoin.N.Sign == Math.Sign(q-2*p), "Certificat produit d=5");
    if (80*p <= 37*q) Verifier(best5[i].N >= 0, "Borne de positivité d=5");
    Console.WriteLine($"alpha={Fraction(Reduire(p,q)),-6} tirage={Decimal(best5[i])}  témoin={Decimal(temoin)}  {Zone(p,q)}");
}

alpha=3/10   tirage=0.819650  témoin=0.196000  borne démontrée


alpha=2/5    tirage=0.765076  témoin=0.072000  borne démontrée


alpha=37/80  tirage=0.730982  témoin=0.021668  borne démontrée


alpha=12/25  tirage=0.721505  témoin=0.010816  zone ouverte


alpha=1/2    tirage=0.710732  témoin=0.000000  zone ouverte


alpha=11/20  tirage=0.684073  témoin=-0.020250  témoin négatif


### Exercice 1 — Deux copies

Adaptez les contractions à deux sites. Calculez les quatre contributions et échantillonnez des matrices de rang au plus deux. À quel $\alpha$ le meilleur tirage change-t-il de signe ? Expliquez pourquoi ce changement ne suffit pas à situer le seuil exact.

In [9]:
// TODO etudiant : calculer la forme sur les deux sites puis relever le minimum observé.
Console.WriteLine("Exercice 1 à compléter");

Exercice 1 à compléter


### Exercice 2 — Optimiser les facteurs

Maintenez le rang par la paramétrisation $XY^*$. Approchez le gradient par différences finies et comparez deux trajectoires sur 200 pas : un départ aléatoire et un départ perturbé proche du témoin. Quelle partie du calcul exact réutiliseriez-vous pour vérifier le dernier itéré ?

In [10]:
// TODO etudiant : renvoyer deux trajectoires de la forme normalisée, une par initialisation.
Console.WriteLine("Exercice 2 à compléter");

Exercice 2 à compléter


### Exercice 3 — Changer d'échelle au seuil une-copie

Pour $d=3,\ldots,12$, convertissez le seuil de Dür $3(d-1)/(4d-2)$ en paramètre $\alpha$ et comparez-le exactement à $1/2$. Justifiez le résultat pour tout $d\ge 2$ par simplification algébrique.

In [11]:
// TODO etudiant : produire un tableau des fractions obtenues et expliquer le résultat.
Console.WriteLine("Exercice 3 à compléter");

Exercice 3 à compléter


## Conclusion et périmètre de parité

Les deux versions calculent la conversion Werner, les contractions à trois copies et la courbe du témoin sur la même grille. La version C# contrôle par égalités rationnelles le seuil et les valeurs du témoin ; les décimales n'interviennent que pour afficher les résultats. **Les minima échantillonnés ne sont pas comparables terme à terme** : le Python utilise des facteurs gaussiens complexes, le C# des facteurs entiers complexes. Dans les deux versions, le témoin fournit un certificat lorsque sa valeur est négative, alors qu'un ensemble de tirages positifs ne résout pas la zone ouverte.